
### début du fine tuning



In [1]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

In [3]:
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

print("Train DataFrame head:")
display(train_df.head())

print("\nTest DataFrame head:")
display(test_df.head())

Train DataFrame head:


,description,patient,doctor
0,are pain and stiffness in the neck symptoms of...,i strained my muscle in my left shoulder back ...,"hi, this is possibly a slipped disc at the lev..."
1,q. what is causing pain during sex at a partic...,"hello doctor, whenever my fiance has been havi...","hi, welcome to iclinq.com. you seem to have pr..."
2,is my abdominal pain due to hiatal hernia or d...,73 year old female. developed left mid abdomin...,"hi, noted the history related to a 73 year old..."
3,trying to concieve. got mirena removed. had bl...,i am a 23 year old female. i would like to hav...,"hi, thanks for the query. after removal of int..."
4,"tested hepatitis b positive, feel fatigued, ha...",i have hepatitis b positive. hepatitis b first...,"hi and thanks for the query,the first thing to..."



Test DataFrame head:


,description,patient,doctor
0,q. i get intermittent abdominal pain before bo...,"hello doctor, i have had intermittent pain in ...","hello, welcome to icliniq.com. as i read your ..."
1,suggest remedy for toothache,hello doctor im having toothache and i took 6g...,hi .. dont worry stop those medicine... have h...
2,q. i cannot read the fourth line on the eye te...,"hello doctor,i have done my eye test. during t...","hello. in order to answer you, i need little m..."
3,what precaution should be taken as having one ...,"hello, i'm a 23 year old male. i just went thr...","hi, you can check for thyroid and testosterone..."
4,"have upper abdominal pain, loose motion with m...",have had upper abdominal and back pain going i...,"watery diarrhoea and mucus passage, since how ..."


## Chargement du modèle Phi-3.5 avec QLoRA

Nous allons charger le modèle `microsoft/Phi-3-mini-4k-instruct` en utilisant la technique QLoRA (Quantized LoRA) pour réduire l'utilisation de la mémoire tout en permettant un fine-tuning efficace.

In [4]:

# Définir l'ID du modèle
model_id = "microsoft/Phi-3-mini-4k-instruct"


In [5]:
# Configuration de la quantisation 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)


In [6]:
from transformers import AutoConfig

# Charger le tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Charger la configuration officielle (trust_remote_code=False par défaut pour utiliser l'intégration transformers)
config = AutoConfig.from_pretrained(model_id, trust_remote_code=False)

# Charger le modèle en utilisant l'implémentation native de transformers
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager"
)

model.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Embedding(32011, 3072, padding_idx=32000)

In [7]:
# Préparer le modèle pour l'entraînement en K-bit
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# Configuration LoRA
lora_config = LoraConfig(
    r=16, # Rang de l'adaptateur LoRA
    lora_alpha=16, # Scaling LoRA alpha
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Modules à fine-tuner
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Ajouter les adaptateurs LoRA au modèle
model = get_peft_model(model, lora_config)

print("Modèle chargé et configuré avec QLoRA :")
model.print_trainable_parameters()

Modèle chargé et configuré avec QLoRA :
trainable params: 8,912,896 || all params: 3,829,666,816 || trainable%: 0.2327


## Test du modèle avant le Fine-Tuning

Nous allons évaluer le comportement du modèle `Phi-3.5` avec ses poids pré-entraînés en générant du texte à partir d'une invite simple.

In [8]:
from transformers import pipeline

# Initialiser le pipeline de génération
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Sélectionner un exemple du dataset de test
sample_index = 0
sample_text = test_df.iloc[sample_index]['description']

# Formater le prompt (Phi-3 utilise un format instruct spécifique)
prompt = f"<|user|>\n{sample_text}<|end|>\n<|assistant|>"

# Paramètres de génération
generation_args = {
    "max_new_tokens": 100,
    "return_full_text": False,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.95,
    "pad_token_id": tokenizer.pad_token_id
}

# Générer la réponse
output = gen(prompt, **generation_args)

print(f"Description du test (index {sample_index}):\n{sample_text}\n")
print(f"Réponse du modèle non-entraîné:\n{output[0]['generated_text']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece

Description du test (index 0):
q. i get intermittent abdominal pain before bowel movement. why?

Réponse du modèle non-entraîné:
Intermittent abdominal pain before a bowel movement can be caused by a variety of conditions. Here are some common reasons:

1. **Constipation**: Difficulty passing stool can lead to pain as the bowel muscles strain.

2. **Irritable Bowel Syndrome (IBS)**: A disorder affecting the large intestine that can cause abdominal pain, bloating, and changes


## Préparation des données et Fine-Tuning

Nous allons transformer le dataset au format d'instruction requis et lancer l'entraînement avec QLoRA.

In [9]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Configuration selon vos précisions :
# Entrée : 'description' (question brute)
# Cible : 'patient' (interprétation/langage naturel)
input_col = 'description'
target_col = 'doctor'

print(f"Entraînement configuré : {input_col} -> {target_col}")

# Fonction pour formater les données au format ChatML
def format_instruction(sample):
	return {"text": f"<|user|>\n{sample[input_col]}<|end|>\n<|assistant|>\n{sample[target_col]}<|end|>"}

# Conversion des DataFrames en Datasets Hugging Face
train_dataset = Dataset.from_pandas(train_df[[input_col, target_col]])
test_dataset = Dataset.from_pandas(test_df[[input_col, target_col]])

# Application du formatage
train_dataset = train_dataset.map(format_instruction, remove_columns=train_dataset.column_names)
test_dataset = test_dataset.map(format_instruction, remove_columns=test_dataset.column_names)

Entraînement configuré : description -> doctor


Map:   0%|          | 0/197230 [00:00<?, ? examples/s]

Map:   0%|          | 0/49308 [00:00<?, ? examples/s]

In [10]:
from trl import SFTTrainer, SFTConfig
import torch

# Configuration de l'entraînement
# On passe max_seq_length via dataset_kwargs pour contourner les erreurs de signature
sft_config = SFTConfig(
    output_dir="./phi-3-qlora-med",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=30,
    save_steps=15,
    optim="paged_adamw_32bit",
    bf16=True,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    report_to="none",
    dataset_text_field="text",
    dataset_kwargs={"max_seq_length": 512},
    packing=False
)

# Initialisation du Trainer sans argument conflictuel direct
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=sft_config,
    processing_class=tokenizer
)

# Lancement du fine-tuning
print("Début de l'entraînement (30 étapes)...")
trainer.train()

# Sauvegarde finale du modèle et du tokenizer
print("Sauvegarde du modèle...")
trainer.save_model("./phi-3-qlora-med-final")
tokenizer.save_pretrained("./phi-3-qlora-med-final")
print("Entraînement et sauvegarde terminés.")

Adding EOS to train dataset:   0%|          | 0/197230 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/197230 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/197230 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/49308 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/49308 [00:00<?, ? examples/s]

...

Step,Training Loss
5,2.886380
10,2.714298
15,2.579264
20,2.423264
25,2.478882
30,2.486756


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Sauvegarde du modèle...


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Entraînement et sauvegarde terminés.
